# Which composition of Omega detects best in finance

This paper computes `Omega = C*D*Coex/M`. Definition 1 of the framework paper
computes `Omega = C*D/(M*Coex)`. They differ only in where `Coex` goes.

This notebook compares them **under the same protocol the paper uses**: 20-day
windows with step 3, the asymmetric persistence filter (0.6 / 0.08), F1 at the
90th percentile against labelled crisis windows, and a block bootstrap. Prices
are downloaded from yfinance, so nothing has to be uploaded.

A third composition is also measured: the **shape-normalised** variant of
Proposition 7 of the framework paper, which replaces `Var(k)` by the Gini
coefficient of the degree sequence and `D` by the mean degree. It matters
because `Var(k)` carries units of degree squared and therefore rises with the
LEVEL of correlation, not only with its shape.

**Prior hypothesis, written before looking at anything:** in simulation all four
factors rise (or `1/M` rises) on the move from calm to crisis, so the
composition used here aligns them and the one in Definition 1 sets `Coex`
against the rest. If that holds on real data, the composition used here should
win.


## 1. Data: banks 2006-2011, the crisis of 2008


In [ ]:
# =============================================================================
# WHICH COMPOSITION DETECTS BEST: the one used here or Definition 1 of the framework
#
# This paper computes            Omega = C*D*Coex / M
# Definition 1 of the framework  Omega = C*D / (M*Coex)
# They differ only in where Coex goes. This notebook compares them under the SAME
# protocol the paper uses: 20-day windows with step 3, the asymmetric filter
# (0.6 / 0.08), F1 at the 90th percentile against labelled crisis windows, and
# a block bootstrap for significance.
#
# A THIRD composition is also measured: the shape-normalised variant of
# Proposition 7, which replaces Var(k) by the Gini of the degree sequence and D
# by the mean degree. It matters because Var(k) carries units of degree squared
# and therefore rises with the LEVEL of correlation, not only with its shape.
#
# PRIOR HYPOTHESIS, written before seeing the result: in simulation all four
# factors rise (or 1/M rises) on the move from calm to crisis, so the
# composition used here aligns them and Definition 1 sets Coex against the rest.
# If that holds on real data, the composition used here should win.
# =============================================================================
import numpy as np, pandas as pd, yfinance as yf

TICKERS = ['JPM','BAC','C','WFC','GS','MS','USB','PNC','BK','STT','AXP','COF']
START, END = '2006-01-01', '2011-12-31'
WIN, STEP = 20, 3
ALPHA_UP, ALPHA_DOWN = 0.6, 0.08
CRISIS = [("2007-08-01", "2009-06-01")]
EPS = 1e-12

px = {}
for t in TICKERS:
    try:
        d = yf.download(t, start=START, end=END, progress=False, auto_adjust=True)
        if len(d) > 500: px[t] = d['Close'].squeeze()
    except Exception as e:
        print('  fallo', t, e)
wide = pd.DataFrame(px).dropna()
rets = np.log(wide).diff().dropna()
n = wide.shape[1]
print(f'{n} tickers, {len(rets)} dias: {rets.index[0].date()} a {rets.index[-1].date()}')

## 2. The four factors and the three compositions


In [ ]:
def gini(x):
    x = np.sort(np.asarray(x, float)); x = x[x >= 0]
    if len(x) == 0 or x.sum() == 0: return 0.0
    N = len(x); cum = np.cumsum(x)
    return (N + 1 - 2 * (cum.sum() / cum[-1])) / N

def factors(A):
    N = A.shape[0]; A2 = A @ A
    C = np.trace(A2 @ A) / (A2.sum() - np.trace(A2) + EPS)
    D = A.sum() / (N * (N - 1))
    k = A.sum(1)
    lam = np.sort(np.linalg.eigvalsh(np.diag(k) - A))
    M = 1.0 / max(lam[1], 1e-6)               # inverse spectral gap, in both papers
    return C, D, M, k.var(), k.mean(), gini(k)

def absorption_ratio(win, k):
    ev = np.sort(np.linalg.eigvalsh(np.cov(win.T)))[::-1]
    return ev[:k].sum() / ev.sum()

k_ar = max(1, round(n / 5))
rows = []
for i in range(WIN, len(rets), STEP):
    w = rets.iloc[i - WIN:i]
    A = np.abs(w.corr().values); np.fill_diagonal(A, 0.0)
    C, D, M, Coex, kbar, Gk = factors(A)
    rows.append(dict(
        date=rets.index[i],
        fin   = C * D * Coex / M,        # the composition used in this paper
        fsri  = C * D / (M * Coex),      # Definition 1 of the framework paper
        shape = C * kbar / (M * Gk),     # shape-normalised (Proposition 7)
        AR    = absorption_ratio(w.values, k_ar)))
R = pd.DataFrame(rows)
R['crisis'] = R.date.apply(lambda x: any(pd.Timestamp(s) <= x <= pd.Timestamp(e)
                                          for s, e in CRISIS))
print(f'{len(R)} windows, {R.crisis.sum()} in crisis ({100*R.crisis.mean():.0f}%)')

## 3. Detection and comparison


In [ ]:
def persistence_filter(x, a_up=ALPHA_UP, a_down=ALPHA_DOWN):
    out = np.zeros_like(x, float); out[0] = x[0]
    for i in range(1, len(x)):
        a = a_up if x[i] > out[i-1] else a_down
        out[i] = a * x[i] + (1 - a) * out[i-1]
    return out

def f1_at_pct(x, crisis, pct=0.90):
    thr = np.quantile(x, pct); al = x >= thr
    tp = (al & crisis).sum(); fp = (al & ~crisis).sum(); fn = (~al & crisis).sum()
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    return 2*p*r/(p+r) if p+r else 0

def bootstrap(a, b, c, B=2000, block=20, seed=0):
    rng = np.random.default_rng(seed); N = len(a); nb = N // block; d = []
    for _ in range(B):
        idx = np.concatenate([np.arange(s, min(s+block, N))
                              for s in rng.integers(0, N-block+1, nb)])
        d.append(f1_at_pct(a[idx], c[idx]) - f1_at_pct(b[idx], c[idx]))
    d = np.array(d)
    return d.mean(), np.quantile(d, .025), np.quantile(d, .975)

c = R.crisis.values
for col in ('fin', 'fsri', 'shape', 'AR'):
    R[col + '_f'] = persistence_filter(R[col].values)

print('DETECTION (F1 at the 90th percentile against labelled crisis windows)')
print(f"{'composition':28}{'raw':>9}{'filtered':>11}")
for col, nom in [('fin','this paper C*D*Coex/M'), ('fsri','Definition 1 C*D/(M*Coex)'),
                 ('shape','shape-normalised'), ('AR','Absorption Ratio (benchmark)')]:
    print(f'  {nom:26}{f1_at_pct(R[col].values, c):>9.3f}{f1_at_pct(R[col+"_f"].values, c):>11.3f}')

print()
print('PAIRED COMPARISONS with a block bootstrap (95% CI)')
for a, b, nom in [('fin','fsri','this paper - Definition 1'),
                  ('fin','shape','this paper - shape-normalised'),
                  ('fin','AR','this paper - Absorption Ratio'),
                  ('fsri','AR','Definition 1 - Absorption Ratio')]:
    d, lo, hi = bootstrap(R[a+'_f'].values, R[b+'_f'].values, c)
    ver = 'first wins' if lo > 0 else ('second wins' if hi < 0 else 'tie')
    print(f'  {nom:30} {d:+.3f}  [{lo:+.3f}, {hi:+.3f}]  {ver}')

print()
print('HOW TO READ IT')
print('  If the composition used here beats Definition 1 with a CI that does not')
print('  cross zero, that composition is justified in the paper by this comparison,')
print('  and it is also explained: under stress the four factors move in the same')
print('  direction, and this composition aligns them.')
print('  If they tie, the choice becomes one of consistency with the framework')
print('  and unifying on Definition 1 is preferable.')
print('  If Definition 1 wins, the index in this paper should be changed.')
print()
print('  Look at the shape-normalised variant too: if it beats both, the effect the')
print('  others detect is partly one of SCALE (Var(k) rises with the level of')
print('  correlation, not only with its shape) and that has to be declared.')
R.to_csv('compositions.csv', index=False)
print('\nWritten to compositions.csv')